# HASTS206 Applied Demographic Analysis: Assignment 2: Part 0 (Setup and data quality)
**University of Zimbabwe, Department of Mathematics and Computational Sciences**

> **All figures in this notebook are HYPOTHETICAL.** The data are synthetic and the district labels are illustrative.

**How to run:** place the original CSVs in `data/`, then run all cells from the repository root (`Kernel > Restart & Run All`). All paths are relative. Outputs consumed by Power BI are written to `outputs/`.

| Section | Content |
|---|---|
| 0 | Setup and data-quality log |
| 1 | Population structure and mortality (pyramids, ratios, balance, rates, abridged life table) |
| 2 | Population projections (arithmetic, geometric, cohort-component, sensitivity) |
| 3 | Three-state continuous-time Markov model (H, I, D) |
| 4 | District clustering (k-means) |
| 5 | Exports for Power BI, reconciliation figures |
| 6 | What-if cell for the recorded presentation |

Each block of code is followed by a **Notes** cell that interprets what was just executed.

In [1]:
# What this cell does: Loads the Python libraries and the five CSV files, sets the folder paths (data
# in, outputs out) and prints how many rows each file has.
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from scipy.linalg import expm
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.options.display.float_format = lambda v: f'{v:,.0f}' if float(v).is_integer() else f'{v:,.4f}'
pd.options.display.width = 200
pd.options.display.max_columns = 40
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .3, 'axes.spines.top': False, 'axes.spines.right': False})

DATA, OUT = Path('data'), Path('outputs')
FIG = OUT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
HYP = '(hypothetical data)'
f0 = lambda x: f'{x:,.0f}'; f1 = lambda x: f'{x:,.1f}'; f2 = lambda x: f'{x:,.2f}'; f3 = lambda x: f'{x:,.3f}'; f4 = lambda x: f'{x:,.4f}'; pc = lambda x: f'{100*x:.1f}%'   # formatting helpers used in the Notes cells
KEY = {}          # key results saved to outputs/key_results.json for the PDF report

districts = pd.read_csv(DATA / 'districts.csv')
age_sex   = pd.read_csv(DATA / 'age_sex.csv')
life_in   = pd.read_csv(DATA / 'life_table_inputs.csv')
scen      = pd.read_csv(DATA / 'projection_scenarios.csv')
ins       = pd.read_csv(DATA / 'insured_transitions.csv')

AGES = sorted(age_sex.age_start.unique())
AGE_LAB = {a: (f'{a}-{a+4}' if a < 85 else '85+') for a in AGES}
for n, d in [('districts', districts), ('age_sex', age_sex), ('life_table_inputs', life_in), ('projection_scenarios', scen), ('insured_transitions', ins)]:
    print(f'{n:22s} rows={len(d):4d} cols={d.shape[1]}')

districts              rows=  24 cols=10
age_sex                rows= 864 cols=9
life_table_inputs      rows=  18 cols=5
projection_scenarios   rows=   3 cols=6
insured_transitions    rows=  96 cols=6


**Notes: loading the data.** Nothing is analysed yet. We load the five CSV files exactly as supplied and print their sizes. The counts are what we expected: 24 districts, 864 age-sex rows (24 districts x 18 age bands x 2 sexes), 18 life-table rows, 3 scenarios and 96 insured rows (24 districts x 4 possible moves). The small helper functions (f0, f1, pc and so on) only format numbers so the notes below can quote results exactly.

## 0. Data-quality log
The original CSVs are never overwritten. Every check below is recorded in a log (saved to `outputs/data_quality_log.csv`) together with the cleaning decision taken in code.

In [2]:
# What this cell does: Runs about 35 data-quality checks (row counts, keys, joins, age and sex
# coverage, blanks, valid ranges, reconciliation, repeated Markov fields) and stores each result and
# decision in a log table.
log = []
def check(area, test, result, status, action=''):
    log.append(dict(area=area, test=test, result=str(result), status=status, action=action))
ok = lambda b: 'PASS' if b else 'FAIL'

# --- row counts and keys
exp = {'districts': 24, 'age_sex': 24 * 18 * 2, 'life_table_inputs': 18, 'projection_scenarios': 3, 'insured_transitions': 24 * 2 * 2}
for n, d in [('districts', districts), ('age_sex', age_sex), ('life_table_inputs', life_in), ('projection_scenarios', scen), ('insured_transitions', ins)]:
    check('Row count', n, f'{len(d)} (expected {exp[n]})', ok(len(d) == exp[n]))
check('Key', 'districts.district_id unique', districts.district_id.is_unique, ok(districts.district_id.is_unique))
check('Key', 'age_sex (district_id, age_start, sex) unique', not age_sex.duplicated(['district_id', 'age_start', 'sex']).any(), ok(not age_sex.duplicated(['district_id', 'age_start', 'sex']).any()))
check('Key', 'life_table age_start unique', life_in.age_start.is_unique, ok(life_in.age_start.is_unique))
check('Key', 'projection_scenarios scenario unique', scen.scenario.is_unique, ok(scen.scenario.is_unique))
check('Key', 'insured (district_id, origin, destination) unique', not ins.duplicated(['district_id', 'origin', 'destination']).any(), ok(not ins.duplicated(['district_id', 'origin', 'destination']).any()))

# --- join cardinality: age_sex (many) -> districts (one)
j = age_sex.merge(districts[['district_id']], on='district_id', how='left', validate='many_to_one', indicator=True)
check('Join', 'age_sex -> districts is many-to-one, no orphans, no row inflation', f'{len(j)} rows after join, orphans={(j._merge != "both").sum()}', ok(len(j) == len(age_sex) and (j._merge == 'both').all()))
per_d = age_sex.groupby('district_id').size()
check('Join', 'rows per district in age_sex', f'min={per_d.min()}, max={per_d.max()}', ok((per_d == 36).all()))
ji = ins.merge(districts[['district_id']], on='district_id', how='left', validate='many_to_one')
check('Join', 'insured_transitions -> districts many-to-one (4 rows per district)', f'{len(ji)} rows; rows per district = {ins.groupby("district_id").size().unique().tolist()}', ok(len(ji) == len(ins)))

# --- coverage
cov = age_sex.groupby('district_id').agg(bands=('age_start', 'nunique'), sexes=('sex', 'nunique'))
check('Coverage', 'age bands 0..85 (18) and sexes F,M in every district', f'bands={cov.bands.unique().tolist()}, sexes={cov.sexes.unique().tolist()}', ok((cov.bands == 18).all() and (cov.sexes == 2).all()))
check('Coverage', 'age_start values', AGES, ok(AGES == list(range(0, 90, 5))))
check('Coverage', 'life-table bands match age_sex bands', life_in.age_start.tolist() == AGES, ok(life_in.age_start.tolist() == AGES))

# --- missing indicators
miss = pd.concat([d.isna().sum().rename(n) for n, d in [('districts', districts), ('age_sex', age_sex), ('life', life_in), ('scen', scen), ('ins', ins)]], axis=1)
miss_list = {c: districts.loc[districts[c].isna(), 'district_id'].tolist() for c in districts.columns if districts[c].isna().any()}
check('Missing', 'districts.piped_water_pct missing', miss_list, 'FLAG', 'Left blank in raw data. Imputed ONLY in the clustering step (Section 4) with an explicit flag column; sensitivity checked.')
check('Missing', 'age_sex, scenarios, insured: any blanks', int(age_sex.isna().sum().sum() + scen.isna().sum().sum() + ins.isna().sum().sum()), ok(age_sex.isna().sum().sum() + scen.isna().sum().sum() + ins.isna().sum().sum() == 0))
lt_blank = life_in[life_in.interval_years.isna() | life_in.ax.isna()].age_start.tolist()
check('Missing', 'life table blanks (interval_years, ax)', f'only at age_start={lt_blank}', ok(lt_blank == [85]), 'Structural (open interval). Handled via L85+ = l85/m85+.')

# --- valid ranges
num_cols = ['population_2025', 'baseline_deaths_2025', 'arrivals_2025', 'departures_2025', 'births_to_women_2025']
check('Range', 'age_sex counts non-negative integers', bool((age_sex[num_cols] >= 0).all().all()), ok((age_sex[num_cols] >= 0).all().all()))
check('Range', 'survival_5yr in (0,1]', f'{age_sex.survival_5yr.min():.4f} to {age_sex.survival_5yr.max():.4f}', ok(age_sex.survival_5yr.between(0, 1).all()))
check('Range', 'baseline deaths <= population', bool((age_sex.baseline_deaths_2025 <= age_sex.population_2025).all()), ok((age_sex.baseline_deaths_2025 <= age_sex.population_2025).all()))
fb = age_sex[age_sex.births_to_women_2025 > 0]
check('Range', 'births only in female rows, ages 15-49', f'male rows with births={(age_sex[(age_sex.sex=="M")].births_to_women_2025>0).sum()}; ages with births={sorted(fb.age_start.unique())}', ok((fb.sex == 'F').all() and fb.age_start.between(15, 45).all()))
check('Range', 'piped_water_pct within 0-100', f'{districts.piped_water_pct.min()} to {districts.piped_water_pct.max()}', ok(districts.piped_water_pct.dropna().between(0, 100).all()))
check('Range', 'life table ax within (0, interval)', 'all 5-year bands', ok(((life_in.ax > 0) & (life_in.ax < life_in.interval_years)).iloc[:-1].all()))
check('Range', 'scenario multipliers positive, newborn survival in (0,1)', 'Low<Base<High ordering holds for fertility, arrivals, newborn survival', ok(scen.fertility_multiplier.is_monotonic_increasing and scen.arrivals_multiplier.is_monotonic_increasing and scen.departures_multiplier.is_monotonic_decreasing))

# --- reconciliation of totals and events
agg = age_sex.groupby('district_id').agg(pop2025=('population_2025', 'sum'), base_deaths=('baseline_deaths_2025', 'sum'),
        arr=('arrivals_2025', 'sum'), dep=('departures_2025', 'sum'), births_f=('births_to_women_2025', 'sum'))
rec = districts.merge(agg, left_on='district_id', right_index=True, validate='one_to_one')
rec['recon_2026'] = rec.pop2025 + rec.births_2025 - rec.deaths_2025 + rec.arrivals_2025 - rec.departures_2025
rec['balance_gap'] = rec.population_2026 - rec.recon_2026
rec['newborn_deaths_implied'] = rec.deaths_2025 - rec.base_deaths
check('Reconcile', 'births: districts vs sum of age_sex female rows', f'max abs diff = {(rec.births_2025 - rec.births_f).abs().max()}', ok((rec.births_2025 == rec.births_f).all()))
check('Reconcile', 'arrivals: districts vs age_sex', f'max abs diff = {(rec.arrivals_2025 - rec.arr).abs().max()}', ok((rec.arrivals_2025 == rec.arr).all()))
check('Reconcile', 'departures: districts vs age_sex', f'max abs diff = {(rec.departures_2025 - rec.dep).abs().max()}', ok((rec.departures_2025 == rec.dep).all()))
check('Reconcile', 'balancing equation: P2025 + B - D + I - E = reported P2026', f'max abs gap = {rec.balance_gap.abs().max()} persons', ok((rec.balance_gap == 0).all()))
check('Reconcile', 'district deaths vs baseline deaths (scope difference)', f'district deaths exceed baseline deaths in {(rec.newborn_deaths_implied>0).sum()}/24 districts; total gap = {rec.newborn_deaths_implied.sum():,}', 'EXPLAINED',
      'Baseline deaths cover only residents present on 1 Jan 2025 (classified by baseline age). District deaths also include deaths among 2025 births. The gap is therefore the implied deaths of 2025 births and must NOT be treated as an error.')

# --- Markov file repeated-field conventions
rep = ins.groupby(['district_id', 'origin']).agg(n_exp=('exposure_py', 'nunique'), n_init=('initial_lives', 'nunique'), rows=('destination', 'size'))
check('Markov file', 'exposure_py and initial_lives repeated identically across the 2 destinations', f'unique values per district-origin: exposure={rep.n_exp.unique().tolist()}, initial={rep.n_init.unique().tolist()}, rows={rep.rows.unique().tolist()}', ok((rep.n_exp == 1).all() and (rep.n_init == 1).all()),
      'Count exposure and initial_lives ONCE per district-origin (drop_duplicates) to avoid double counting.')
check('Markov file', 'origins/destinations valid (H->I,D ; I->H,D)', sorted(set(zip(ins.origin, ins.destination))), ok(set(zip(ins.origin, ins.destination)) == {('H', 'I'), ('H', 'D'), ('I', 'H'), ('I', 'D')}))
check('Markov file', 'counts non-negative; exposure positive', f'min exposure={ins.exposure_py.min()}', ok((ins.transition_count >= 0).all() and (ins.exposure_py > 0).all()))
check('Cleaning', 'Original CSVs unchanged; no rows dropped', 'All derived tables are built in memory and exported separately to outputs/', 'DONE')

dq = pd.DataFrame(log)
dq.to_csv(OUT / 'data_quality_log.csv', index=False)
KEY['dq_pass'] = int((dq.status == 'PASS').sum()); KEY['dq_total'] = len(dq); KEY['dq_fail'] = int((dq.status == 'FAIL').sum())
dq

,area,test,result,status,action
0,Row count,districts,24 (expected 24),PASS,
1,Row count,age_sex,864 (expected 864),PASS,
2,Row count,life_table_inputs,18 (expected 18),PASS,
3,Row count,projection_scenarios,3 (expected 3),PASS,
4,Row count,insured_transitions,96 (expected 96),PASS,
5,Key,districts.district_id unique,True,PASS,
6,Key,"age_sex (district_id, age_start, sex) unique",True,PASS,
7,Key,life_table age_start unique,True,PASS,
8,Key,projection_scenarios scenario unique,True,PASS,
9,Key,"insured (district_id, origin, destination) unique",True,PASS,


**Notes: data-quality log.**
- 32 of 35 checks pass and 0 fail. The rest are flags that we dealt with on purpose: two blank water values (D04, D17), the blank interval and a_x at the open 85+ group, and the deaths scope difference.
- Keys are unique. The age-sex file joins to the district file many-to-one, with no orphan rows and no extra rows after the join (864 before and after). Every district has all 18 age bands for both sexes.
- The Markov file repeats exposure and initial lives on both destination rows of each district-origin pair. We confirmed that, and we count each of them once later on.
- **Decision:** nothing is deleted or changed in the original CSVs. The missing water values stay blank here and are imputed later, only for clustering, with a flag.

In [3]:
# What this cell does: Rebuilds the 2026 population in every district from the balancing equation and
# compares it with the reported 2026 figure. It also compares district deaths with baseline-file
# deaths.
# Population-balance reconciliation for all 24 districts, plus the deaths scope difference
rec[['district_id', 'pop2025', 'births_2025', 'deaths_2025', 'arrivals_2025', 'departures_2025', 'recon_2026', 'population_2026', 'balance_gap', 'base_deaths', 'newborn_deaths_implied']].assign(
    newborn_deaths_per_100_births=lambda x: 100 * x.newborn_deaths_implied / x.births_2025).round(2)

,district_id,pop2025,births_2025,deaths_2025,arrivals_2025,departures_2025,recon_2026,population_2026,balance_gap,base_deaths,newborn_deaths_implied,newborn_deaths_per_100_births
0,D01,172445,3939,2166,2460,2926,173752,173752,0,2119,47,1.1900
1,D02,112211,2480,1173,2705,1618,114605,114605,0,1143,30,1.2100
2,D03,77907,1008,1639,1278,1458,77096,77096,0,1627,12,1.1900
3,D04,95654,2268,1239,1303,1655,96331,96331,0,1212,27,1.1900
4,D05,152444,3469,1597,3910,1749,156477,156477,0,1555,42,1.2100
5,D06,59525,774,1217,988,1063,59007,59007,0,1208,9,1.1600
6,D07,44047,935,552,581,682,44329,44329,0,541,11,1.1800
7,D08,55286,1264,580,1416,712,56674,56674,0,565,15,1.1900
8,D09,107511,1277,2178,1850,2002,106458,106458,0,2163,15,1.1700
9,D10,111451,2405,1372,1559,1664,112379,112379,0,1343,29,1.2100


**Notes: reconciliation and the deaths scope difference.**
- For all 24 districts, opening population + births - deaths + arrivals - departures gives exactly the reported 1 January 2026 population (gap = 0). Births, arrivals and departures in the district file also match the age-sex file.
- **Deaths do not match, and that is correct.** The baseline-file deaths only cover people who were alive and living in the district on 1 January 2025, grouped by their age then. District deaths also count babies born during 2025 who died before the year ended. Those babies were never in the baseline population. The gap is 621 deaths in total, about 1.20 per 100 births. If we used baseline deaths in the balancing equation, the 2026 total would be too high by exactly this amount.

In [4]:
# What this cell does: Writes a data dictionary: for every field, what it means, its unit, its valid
# values and its validation rule. Saves it as data_dictionary.csv.
# Data dictionary (Unit 4: write the dictionary before cleaning; every field, unit and validation rule in one place)
dd = pd.DataFrame([
    ['districts.csv', 'district_id', 'District key D01 to D24', 'Text, 24 unique values', 'One row per district'],
    ['districts.csv', 'population_2020, population_2026', 'Opening stock on 1 January (persons)', 'Positive integers', 'Used for the trend fit and the 2026 comparison'],
    ['districts.csv', 'births_2025, deaths_2025, arrivals_2025, departures_2025', 'Calendar-2025 events (persons). deaths_2025 INCLUDES deaths of babies born in 2025', 'Non-negative integers', 'Must close the balancing equation'],
    ['districts.csv', 'piped_water_pct', 'Households with piped water (%)', '0 to 100; 2 blanks', 'Blank = missing (item non-response); imputed only in clustering'],
    ['districts.csv', 'region', 'Synthetic reporting label', 'Region 1 to 4', 'Not used as a clustering feature'],
    ['age_sex.csv', 'district_id, age_start, sex', 'Key. age_start 0,5,...,85 (85 = 85+); sex F or M', '18 bands x 2 sexes per district', 'Unique key; every district complete'],
    ['age_sex.csv', 'population_2025', 'Persons on 1 January 2025', 'Non-negative integers', 'Summed to give the opening 2025 stock'],
    ['age_sex.csv', 'baseline_deaths_2025', 'Deaths in 2025 among people present at baseline, by baseline age', 'Non-negative, <= population', 'EXCLUDES deaths of 2025 births'],
    ['age_sex.csv', 'arrivals_2025, departures_2025', 'Migration flows by destination age band and sex', 'Non-negative integers', 'Sum equals district totals'],
    ['age_sex.csv', 'births_to_women_2025', 'Births by mother\'s baseline age. Male rows are structural zeros', 'Positive only for female rows aged 15-49', 'Sum over female rows equals district births'],
    ['age_sex.csv', 'survival_5yr', 'Supplied 5-year survival probability (85+ = retention in the open group)', 'Between 0 and 1', 'Used as given'],
    ['life_table_inputs.csv', 'age_start, interval_years, person_years, deaths, ax', 'Pooled-sex mortality schedule; ax = average years lived in the interval by those who die in it', 'Blank interval_years and ax at 85+ only', 'm = deaths / person_years'],
    ['projection_scenarios.csv', 'fertility / arrivals / departures multipliers; newborn_survival_f / m', 'Scenario scalars and survival from birth to end of the 5-year step', 'Positive; survival in (0,1)', 'Low, Base, High'],
    ['insured_transitions.csv', 'district_id, origin, destination', 'Key for the observed H, I, D jumps', 'H to I, H to D, I to H, I to D', 'Unique key'],
    ['insured_transitions.csv', 'exposure_py, initial_lives', 'Repeated on both destination rows of a district-origin pair', 'Positive', 'Count ONCE per district-origin'],
    ['insured_transitions.csv', 'transition_count', 'Observed jumps for that origin-destination pair', 'Non-negative integers', 'Summed over districts'],
], columns=['file', 'field', 'meaning and unit', 'valid values', 'validation rule or note'])
dd.to_csv(OUT / 'data_dictionary.csv', index=False)
dd

,file,field,meaning and unit,valid values,validation rule or note
0,districts.csv,district_id,District key D01 to D24,"Text, 24 unique values",One row per district
1,districts.csv,"population_2020, population_2026",Opening stock on 1 January (persons),Positive integers,Used for the trend fit and the 2026 comparison
2,districts.csv,"births_2025, deaths_2025, arrivals_2025, depar...",Calendar-2025 events (persons). deaths_2025 IN...,Non-negative integers,Must close the balancing equation
3,districts.csv,piped_water_pct,Households with piped water (%),0 to 100; 2 blanks,Blank = missing (item non-response); imputed o...
4,districts.csv,region,Synthetic reporting label,Region 1 to 4,Not used as a clustering feature
5,age_sex.csv,"district_id, age_start, sex","Key. age_start 0,5,...,85 (85 = 85+); sex F or M",18 bands x 2 sexes per district,Unique key; every district complete
6,age_sex.csv,population_2025,Persons on 1 January 2025,Non-negative integers,Summed to give the opening 2025 stock
7,age_sex.csv,baseline_deaths_2025,Deaths in 2025 among people present at baselin...,"Non-negative, <= population",EXCLUDES deaths of 2025 births
8,age_sex.csv,"arrivals_2025, departures_2025",Migration flows by destination age band and sex,Non-negative integers,Sum equals district totals
9,age_sex.csv,births_to_women_2025,Births by mother's baseline age. Male rows are...,Positive only for female rows aged 15-49,Sum over female rows equals district births


**Notes: data dictionary.** This table is the reference for every check above. The three points that matter most are: (1) district deaths include deaths among 2025 births but baseline deaths do not, (2) exposure and initial lives are repeated on both rows of each district-origin pair, and (3) the blank water values and the blank 85+ interval mean different things (item non-response versus a structural blank).

In [5]:
# What this cell does: Counts blanks in every file (missingness table) and records rows read, rows
# kept and rows removed for each file (audit trail).
# Missingness table and audit trail (Unit 4: report valid denominators, keep flags, record before/after counts)
frames = {'districts': districts, 'age_sex': age_sex, 'life_table_inputs': life_in, 'projection_scenarios': scen, 'insured_transitions': ins}
miss_rows = []
for fname, fr in frames.items():
    for col in fr.columns:
        n_na = int(fr[col].isna().sum())
        if n_na:
            miss_rows.append(dict(file=fname, field=col, n_missing=n_na, n_valid=len(fr) - n_na, pct_missing=100 * n_na / len(fr),
                                  type='structural (open interval)' if fname == 'life_table_inputs' else 'item non-response (blank)',
                                  treatment='kept blank; handled by the 85+ formula' if fname == 'life_table_inputs' else 'flagged; imputed in clustering only (water_imputed = True)'))
miss_tbl = pd.DataFrame(miss_rows)
audit = pd.DataFrame([dict(file=f, rows_read=len(fr), rows_after_cleaning=len(fr), rows_removed=0, duplicates_found=0) for f, fr in frames.items()])
miss_tbl.to_csv(OUT / 'missingness_table.csv', index=False); audit.to_csv(OUT / 'audit_trail.csv', index=False)
print('Missing values by field:'); print(miss_tbl.round(1).to_string(index=False))
print('\nAudit trail (nothing removed; originals untouched):'); print(audit.to_string(index=False))

Missing values by field:
             file           field  n_missing  n_valid  pct_missing                       type                                                  treatment
        districts piped_water_pct          2       22       8.3000  item non-response (blank) flagged; imputed in clustering only (water_imputed = True)
life_table_inputs  interval_years          1       17       5.6000 structural (open interval)                     kept blank; handled by the 85+ formula
life_table_inputs              ax          1       17       5.6000 structural (open interval)                     kept blank; handled by the 85+ formula

Audit trail (nothing removed; originals untouched):
                file  rows_read  rows_after_cleaning  rows_removed  duplicates_found
           districts         24                   24             0                 0
             age_sex        864                  864             0                 0
   life_table_inputs         18                   18   

**Notes: missingness and audit trail.** Only 3 fields have any blanks. The water field is missing for 2 of 24 districts (8.3%). That is item non-response, so we flag it and treat it carefully in the clustering. The blanks in interval_years and a_x are structural, because the 85+ group is open-ended and has no width, and they are handled by the L(85+) formula. The audit trail shows that every file has the same number of rows before and after cleaning, so no records were lost.